In [4]:
from pyspark.sql import functions as F

# ==========================================
# 1. ОЧИЩЕННЯ СТАНЦІЙ -> lh_silver.dbo.stations
# ==========================================
city_map = {
    "kyiv": "Kyiv", "kiev": "Kyiv", "київ": "Kyiv", "киев": "Kyiv",
    "lviv": "Lviv", "lvov": "Lviv", "львів": "Lviv", "lwów": "Lviv", "lwow": "Lviv",
    "kraków": "Kraków", "krakow": "Kraków", "cracow": "Kraków", "краків": "Kraków",
    "warsaw": "Warsaw", "warszawa": "Warsaw", "варшава": "Warsaw",
}

country_map = {
    "ua": "UA", "ukraine": "UA", "україна": "UA", "ukraina": "UA",
    "pl": "PL", "poland": "PL", "polska": "PL", "польща": "PL",
}

df_stations_raw = spark.table("lh_bronze.air_quality.stations")

df_stations_clean = (
    df_stations_raw
    .filter(F.col("station_id").isNotNull())
    .withColumn("name", F.trim(F.col("name")))
    .withColumn("city", F.lower(F.trim(F.col("city"))))
    .withColumn("country", F.lower(F.trim(F.col("country"))))
    .replace(city_map, subset=["city"])
    .replace(country_map, subset=["country"])
    .select("station_id", "name", "city", "country")
    .dropDuplicates(["station_id"])
)

df_stations_clean.write.format("delta").mode("overwrite").saveAsTable("dbo.stations")

# ==========================================
# 2. ОЧИЩЕННЯ СЕНСОРІВ -> lh_silver.dbo.sensors
# ==========================================
metric_map = {
    "pm2.5": "PM2.5", "pm25": "PM2.5", "pm2,5": "PM2.5",
    "pm10": "PM10",
    "co2": "CO2", "co₂": "CO2",
    "temperature": "temperature", "temp": "temperature", "t": "temperature",
    "humidity": "humidity", "hum": "humidity", "rh": "humidity",
}

unit_map = {
    "PM2.5": "µg/m3",
    "PM10": "µg/m3",
    "CO2": "ppm",
    "temperature": "°C",
    "humidity": "%",
}

df_sensors_raw = spark.table("lh_bronze.air_quality.sensors")

df_sensors_clean = (
    df_sensors_raw
    .filter(F.col("sensor_id").isNotNull())
    .withColumn("metric", F.regexp_replace(F.lower(F.trim(F.col("metric"))), r"[\s_]+", ""))
    .replace(metric_map, subset=["metric"])
    .withColumn("unit", F.col("metric"))
    .replace(unit_map, subset=["unit"])
    .select("sensor_id", "station_id", "metric", "unit")
    .dropDuplicates(["sensor_id"])
    .join(df_stations_clean.select("station_id"), on="station_id", how="inner")
    .select("sensor_id", "station_id", "metric", "unit")
)

df_sensors_clean.write.format("delta").mode("overwrite").saveAsTable("dbo.sensors")

display(spark.table("dbo.stations"))
display(spark.table("dbo.sensors"))


StatementMeta(, 526fb98d-b97b-4171-80b6-f768074955e6, 6, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, 8ce7c7cd-5c8e-4e8d-802b-6fb981c3c559)

SynapseWidget(Synapse.DataFrame, c7554db1-76a9-4774-a90f-ba4198568505)

In [5]:
from pyspark.sql import functions as F

# ==========================================
# 3. ОЧИЩЕННЯ ВИМІРІВ -> lh_silver.dbo.readings
# ==========================================
df_readings_raw = spark.table("lh_bronze.air_quality.readings")

# 1. Прибираємо повні дублікати за полями вихідного файла (без _ingested_at)
file_cols = ["reading_id", "sensor_id", "value", "measured_at", "status"]
df_dedup = df_readings_raw.dropDuplicates(file_cols)

# 2. Фільтруємо статус 'ok' (без урахування регістру і пробілів на краях)
df_ok = df_dedup.filter(F.lower(F.trim(F.col("status"))) == "ok")

# 3. Приводимо value до double (десяткову кому замінюємо на крапку, нечислові -> NULL через try_cast)
# Залишаємо лише непорожні та невід'ємні (>= 0) значення
df_val = (
    df_ok
    .withColumn("value_clean", F.regexp_replace(F.trim(F.col("value").cast("string")), ",", "."))
    .withColumn("value", F.expr("try_cast(value_clean AS double)"))
    .filter(F.col("value").isNotNull() & (F.col("value") >= 0))
)

# 4. Розпізнаємо дату й час у measured_at за допомогою try_to_timestamp (підтримка ISO 8601 та локальних форматів)
ts_str = F.trim(F.col("measured_at").cast("string"))

parsed_ts = F.coalesce(
    F.expr("try_cast(measured_at AS timestamp)"),
    F.expr("try_to_timestamp(trim(measured_at), 'yyyy-MM-dd\\'T\\'HH:mm:ssX')"),
    F.expr("try_to_timestamp(trim(measured_at), 'yyyy-MM-dd\\'T\\'HH:mmX')"),
    F.expr("try_to_timestamp(trim(measured_at), 'yyyy-MM-dd HH:mm:ss')"),
    F.expr("try_to_timestamp(trim(measured_at), 'yyyy-MM-dd HH:mm')"),
    F.expr("try_to_timestamp(trim(measured_at), 'yyyy-MM-dd')"),
    F.expr("try_to_timestamp(trim(measured_at), 'dd.MM.yyyy HH:mm:ss')"),
    F.expr("try_to_timestamp(trim(measured_at), 'dd.MM.yyyy HH:mm')"),
    F.expr("try_to_timestamp(trim(measured_at), 'dd.MM.yyyy')"),
    F.expr("try_to_timestamp(trim(measured_at), 'dd/MM/yyyy HH:mm:ss')"),
    F.expr("try_to_timestamp(trim(measured_at), 'dd/MM/yyyy HH:mm')"),
    F.expr("try_to_timestamp(trim(measured_at), 'dd/MM/yyyy')"),
    F.expr("try_to_timestamp(trim(measured_at), 'yyyy/MM/dd HH:mm:ss')"),
    F.expr("try_to_timestamp(trim(measured_at), 'yyyy/MM/dd HH:mm')"),
    F.expr("try_to_timestamp(trim(measured_at), 'yyyy/MM/dd')")
)

df_dates = (
    df_val
    .withColumn("measured_at", parsed_ts)
    .filter(F.col("measured_at").isNotNull())
    .withColumn("measured_date", F.to_date(F.col("measured_at")))
)

# 5. Залишаємо лише покази сенсорів, які є в dbo.sensors, і потрібні колонки у точному порядку
df_sensors_valid = spark.table("dbo.sensors").select("sensor_id")

df_readings_clean = (
    df_dates
    .join(df_sensors_valid, on="sensor_id", how="inner")
    .select("reading_id", "sensor_id", "value", "measured_at", "measured_date")
)

df_readings_clean.write.format("delta").mode("overwrite").saveAsTable("dbo.readings")

display(spark.table("dbo.readings"))


StatementMeta(, 526fb98d-b97b-4171-80b6-f768074955e6, 7, Finished, Available, Finished, False)

SynapseWidget(Synapse.DataFrame, a54c4cb9-34ee-428e-8386-be301b03994c)